# ECG Classification — Phase 3: Hierarchical Inference & Evaluation

Runs the full hierarchical pipeline end-to-end on the held-out test set:

```
Every test window (N, 4500)
        │
        ▼
  [Stage 1 — TCN-ResNet + NeuroKit2 features]
        │
        ├─── P(NonNormal) < threshold  ──►  Predict: Normal (N)
        │
        └─── P(NonNormal) ≥ threshold  ──►  [Stage 2 — EfficientNet-B2 on spectrograms]
                                                    │
                                                    ├──► AF (A)
                                                    ├──► Other (O)
                                                    └──► Noisy (~)
```

**File sources (all from `ECG_Hierarchical/` produced by Phase 1):**

| File | Used for |
|------|----------|
| `X_test_sig.npy` | Stage 1 signal input |
| `X_test_feat.npy` | Stage 1 NeuroKit2 feature input |
| `X_test_spec.dat` | Stage 2 spectrogram input |
| `y_test.npy` | 4-class ground truth labels |
| `y_test_3class_spec.npy` | Stage 2 ground truth (A/O/~) |
| `memmap_shapes.pkl` | Spectrogram array shape metadata |

**Checkpoint sources:**
- `checkpoints_stage1_v2/best_model.pt` — Stage 1 weights
- `checkpoints_stage1_v2/best_threshold.json` — tuned NonNormal threshold
- `checkpoints_stage2/best_model.pt` — Stage 2 weights
- `checkpoints_stage2/stage2_info.json` — Stage 2 class map


## 1 · Imports

In [ ]:
import os, json, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b2
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix,
                              matthews_corrcoef, roc_auc_score)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())


## 2 · Paths and constants

In [ ]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_PATH  = "/content/drive/MyDrive"
DATA_PATH  = os.path.join(BASE_PATH, "ECG_Hierarchical")
S1_CKPT_DIR = os.path.join(BASE_PATH, "checkpoints_stage1_v2")
S2_CKPT_DIR = os.path.join(BASE_PATH, "checkpoints_stage2")
OUT_DIR    = os.path.join(BASE_PATH, "phase3_results")
os.makedirs(OUT_DIR, exist_ok=True)

STAGE1_MODEL     = os.path.join(S1_CKPT_DIR, "best_model.pt")
STAGE1_THRESHOLD = os.path.join(S1_CKPT_DIR, "best_threshold.json")
STAGE2_MODEL     = os.path.join(S2_CKPT_DIR, "best_model.pt")
STAGE2_INFO      = os.path.join(S2_CKPT_DIR, "stage2_info.json")

CLASS_NAMES_4 = ["Normal (N)", "AF (A)", "Other (O)", "Noisy (~)"]
CLASS_NAMES_2 = ["Normal", "NonNormal"]
CLASS_NAMES_3 = ["AF (A)", "Other (O)", "Noisy (~)"]

BATCH_SIZE = 64

print("Device      :", DEVICE)
print("Data path   :", DATA_PATH)
print("Output dir  :", OUT_DIR)


## 3 · Load checkpoint metadata

In [ ]:
with open(STAGE1_THRESHOLD) as f:
    thr_data = json.load(f)
S1_THRESHOLD = thr_data["stage1_threshold"]
print(f"Stage 1 threshold : {S1_THRESHOLD}")
print(f"  (tuned val macro F1: {thr_data.get('macro_f1', '?')}  "
      f"F1_N: {thr_data.get('f1_N', '?')}  F1_NN: {thr_data.get('f1_NN', '?')})")

with open(STAGE2_INFO) as f:
    s2_info = json.load(f)
print(f"\nStage 2 val  macro F1 : {s2_info.get('val_macro_f1',  '?')}")
print(f"Stage 2 test macro F1 : {s2_info.get('test_macro_f1', '?')}")
print(f"  AF={s2_info.get('val_f1_AF','?')}  "
      f"OT={s2_info.get('val_f1_OT','?')}  "
      f"NO={s2_info.get('val_f1_NO','?')}")


## 4 · Load test data from Phase 1 outputs

In [ ]:
with open(os.path.join(DATA_PATH, "memmap_shapes.pkl"), "rb") as f:
    shapes = pickle.load(f)
test_spec_shape = shapes["test_spec"]   # (N, 3, 224, 224)

# Stage 1 inputs
X_test_sig  = np.load(os.path.join(DATA_PATH, "X_test_sig.npy"),  mmap_mode="r")
X_test_feat = np.load(os.path.join(DATA_PATH, "X_test_feat.npy"), mmap_mode="r")

# Stage 2 inputs (non-Normal only, memory-mapped)
X_test_spec = np.memmap(os.path.join(DATA_PATH, "X_test_spec.dat"),
                         dtype="float32", mode="r", shape=test_spec_shape)

# Labels
y_test_4cls = np.load(os.path.join(DATA_PATH, "y_test.npy"))
y_test_s2   = np.load(os.path.join(DATA_PATH, "y_test_3class_spec.npy"))

n_total      = len(y_test_4cls)
n_nonnormal  = int((y_test_4cls != 0).sum())

print(f"Test windows (total)  : {n_total}")
print(f"Test spectrograms     : {X_test_spec.shape}")
print(f"Test features         : {X_test_feat.shape}")
print(f"\n4-class label counts:")
for i, name in enumerate(CLASS_NAMES_4):
    print(f"  {name}: {(y_test_4cls == i).sum()}")
print(f"\nStage 2 label counts (non-Normal only, remapped A→0 O→1 ~→2):")
for i, name in enumerate(CLASS_NAMES_3):
    print(f"  {name}: {(y_test_s2 == i).sum()}")


## 5 · Stage 1 model definition

Must exactly match the architecture used in Phase 2 Stage 1 training:
5-block TCN-ResNet hybrid, SE-1D attention, multi-scale pooling, NeuroKit2 feature fusion.


In [ ]:
# ── SE-1D attention ───────────────────────────────────────────────────────
class SE1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(channels, max(channels // reduction, 4)), nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.gate(x).unsqueeze(-1)

# ── Dilated TCN block ──────────────────────────────────────────────────────
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=9, dilation=1, dropout=0.3):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.utils.weight_norm(
                nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=pad)),
            nn.PReLU(), nn.Dropout(dropout),
            nn.utils.weight_norm(
                nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation, padding=pad)),
            nn.PReLU(), nn.Dropout(dropout),
        )
        self.proj = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        out = self.net(x)
        return out[:, :, :x.size(2)] + self.proj(x)

# ── Feature MLP ───────────────────────────────────────────────────────────
class FeatureMLP(nn.Module):
    def __init__(self, n_feat, hidden=64, out=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out),   nn.LayerNorm(out),    nn.GELU(),
        )
    def forward(self, x):
        return self.net(x)

# ── Full Stage 1 model ─────────────────────────────────────────────────────
class Stage1Model(nn.Module):
    def __init__(self, n_feat, dropout=0.3):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.utils.weight_norm(nn.Conv1d(1, 64, kernel_size=15, padding=7)),
            nn.PReLU(),
        )
        self.blocks = nn.ModuleList([
            TCNBlock(64,  64,  dilation=1,  dropout=dropout),
            TCNBlock(64,  128, dilation=2,  dropout=dropout),
            TCNBlock(128, 256, dilation=4,  dropout=dropout),
            TCNBlock(256, 512, dilation=8,  dropout=dropout),
            TCNBlock(512, 512, dilation=16, dropout=dropout),
        ])
        self.se       = SE1D(512)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.sig_head = nn.Sequential(
            nn.Linear(1024, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
        )
        self.feat_mlp = FeatureMLP(n_feat, hidden=64, out=64, dropout=dropout)
        self.fusion   = nn.Sequential(
            nn.Linear(256 + 64, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 2)
        )
    def forward(self, sig, feat):
        x = sig.unsqueeze(1)
        x = self.input_conv(x)
        for block in self.blocks:
            x = block(x)
        x   = self.se(x)
        avg = self.avg_pool(x).squeeze(-1)
        mx  = self.max_pool(x).squeeze(-1)
        x   = self.sig_head(torch.cat([avg, mx], dim=1))
        f   = self.feat_mlp(feat)
        return self.fusion(torch.cat([x, f], dim=1))


## 6 · Stage 2 model definition

Must exactly match Phase 2 Stage 2: EfficientNet-B2 with upgraded 2-layer MLP classifier head.


In [ ]:
def build_stage2_model(n_classes=3, dropout=0.4):
    model = efficientnet_b2(weights=None)
    in_features = model.classifier[1].in_features   # 1408
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=False),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(p=dropout / 2),
        nn.Linear(512, n_classes),
    )
    return model


## 7 · Load trained weights

In [ ]:
n_feat = X_test_feat.shape[1]

stage1_model = Stage1Model(n_feat=n_feat, dropout=0.3).to(DEVICE)
stage1_model.load_state_dict(torch.load(STAGE1_MODEL, map_location=DEVICE))
stage1_model.eval()
print(f"Stage 1 weights loaded  ({sum(p.numel() for p in stage1_model.parameters()):,} params)")

stage2_model = build_stage2_model(n_classes=3, dropout=0.4).to(DEVICE)
stage2_model.load_state_dict(torch.load(STAGE2_MODEL, map_location=DEVICE))
stage2_model.eval()
print(f"Stage 2 weights loaded  ({sum(p.numel() for p in stage2_model.parameters()):,} params)")


## 8 · Stage 1 inference

Runs on all test windows using both the raw signal and NeuroKit2 features.
A window is predicted **Normal** if P(NonNormal) < `S1_THRESHOLD`, otherwise **NonNormal**.


In [ ]:
def run_stage1_inference(model, X_sig, X_feat, threshold, batch_size=BATCH_SIZE):
    all_probs, all_preds = [], []
    n = len(X_sig)
    model.eval()
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end    = min(start + batch_size, n)
            x_sig  = torch.tensor(np.array(X_sig[start:end]),  dtype=torch.float32).to(DEVICE)
            x_feat = torch.tensor(np.array(X_feat[start:end]), dtype=torch.float32).to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = model(x_sig, x_feat)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend((probs[:, 1] >= threshold).astype(int))
    return np.array(all_preds), np.vstack(all_probs)

print(f"Running Stage 1 inference  (threshold = {S1_THRESHOLD}) ...")
s1_preds, s1_probs = run_stage1_inference(stage1_model, X_test_sig, X_test_feat, S1_THRESHOLD)

s1_true  = (y_test_4cls != 0).astype(int)
s1_f1    = f1_score(s1_true, s1_preds, average="macro")
s1_f1_N  = f1_score(s1_true, s1_preds, labels=[0], average="macro")
s1_f1_NN = f1_score(s1_true, s1_preds, labels=[1], average="macro")
s1_acc   = np.mean(s1_preds == s1_true)

print(f"Stage 1 binary accuracy  : {s1_acc:.4f}")
print(f"Stage 1 macro F1         : {s1_f1:.4f}")
print(f"  Normal F1    : {s1_f1_N:.4f}")
print(f"  NonNormal F1 : {s1_f1_NN:.4f}")
print(f"\n  Predicted Normal    : {(s1_preds == 0).sum()}")
print(f"  Predicted NonNormal : {(s1_preds == 1).sum()}")
print(f"  True Normal         : {(s1_true  == 0).sum()}")
print(f"  True NonNormal      : {(s1_true  == 1).sum()}")


### Stage 1 error breakdown

In [ ]:
s1_cm = confusion_matrix(s1_true, s1_preds)
tn, fp, fn, tp = s1_cm.ravel()
print("Stage 1 confusion matrix (binary):")
print(f"  True Normal  → Normal    (TN): {tn}  ({tn/(tn+fp)*100:.1f}%)")
print(f"  True Normal  → NonNormal (FP): {fp}  ({fp/(tn+fp)*100:.1f}%)  ← false alarm")
print(f"  True NonNorm → Normal    (FN): {fn}  ({fn/(fn+tp)*100:.1f}%)  ← miss (worst error)")
print(f"  True NonNorm → NonNormal (TP): {tp}  ({tp/(fn+tp)*100:.1f}%)")
print(f"  Sensitivity (recall NonNormal): {tp/(fn+tp):.4f}")
print(f"  Specificity (recall Normal)   : {tn/(tn+fp):.4f}")


## 9 · Stage 2 inference

Runs on the non-Normal test spectrograms (generated in Phase 1 from true non-Normal windows).
Returns 3-class probabilities: AF (0), Other (1), Noisy (2).


In [ ]:
def run_stage2_inference(model, X_spec, batch_size=BATCH_SIZE):
    all_probs, all_preds = [], []
    n = len(X_spec)
    model.eval()
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end    = min(start + batch_size, n)
            x_batch = torch.tensor(np.array(X_spec[start:end]), dtype=torch.float32).to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = model(x_batch)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(probs.argmax(axis=1))
    return np.array(all_preds), np.vstack(all_probs)

print("Running Stage 2 inference ...")
s2_preds, s2_probs = run_stage2_inference(stage2_model, X_test_spec)

s2_f1  = f1_score(y_test_s2, s2_preds, average="macro")
s2_acc = np.mean(s2_preds == y_test_s2)

print(f"Stage 2 accuracy  : {s2_acc:.4f}")
print(f"Stage 2 macro F1  : {s2_f1:.4f}")
print(f"  AF    F1 : {f1_score(y_test_s2, s2_preds, labels=[0], average='macro'):.4f}")
print(f"  Other F1 : {f1_score(y_test_s2, s2_preds, labels=[1], average='macro'):.4f}")
print(f"  Noisy F1 : {f1_score(y_test_s2, s2_preds, labels=[2], average='macro'):.4f}")


## 10 · Hierarchical combination

**Alignment note:** `X_test_spec` contains exactly the windows where `y_test_4cls != 0`,
in the same order as they appear in `y_test_4cls`. We build `true_nonnormal_idx` to map
Stage 2 predictions back to the correct positions in the full 4-class array.

**Decision rules:**
- Window predicted **Normal** by Stage 1 → final label = 0 (Normal)
- Window predicted **NonNormal** by Stage 1 AND is truly non-Normal (has a Stage 2 prediction) → Stage 2 decides: AF/Other/Noisy
- Window predicted **NonNormal** by Stage 1 but Stage 1 was wrong (true Normal, no Stage 2 entry) → Stage 1 decision is kept as-is and marked NonNormal; the 4-class label defaults to AF (most common non-Normal class) — these represent irreducible Stage 1 false alarms


In [ ]:
# S2 label remap: 0(AF)→1, 1(Other)→2, 2(Noisy)→3
S2_TO_4CLASS = {0: 1, 1: 2, 2: 3}

final_preds = np.zeros(n_total, dtype=int)
prob_4cls   = np.zeros((n_total, 4), dtype=np.float32)

# Index in the full array where y_test_4cls != 0
true_nonnormal_idx = np.where(y_test_4cls != 0)[0]

# ── Pass 1: fill every window with Stage 1 Normal prediction ──────────────
normal_mask = (s1_preds == 0)
final_preds[normal_mask]  = 0
prob_4cls[normal_mask, 0] = s1_probs[normal_mask, 0]

# ── Pass 2: for each true non-Normal window, apply Stage 2 ───────────────
# Stage 2 predictions are indexed 0..len(true_nonnormal_idx)-1
for j, global_idx in enumerate(true_nonnormal_idx):
    if j >= len(s2_preds):
        break   # safety guard

    s1_says_nonnormal = (s1_preds[global_idx] == 1)

    if s1_says_nonnormal:
        # Stage 1 correct → trust Stage 2 for subtype
        cls4     = S2_TO_4CLASS[int(s2_preds[j])]
        final_preds[global_idx]  = cls4
        # Joint probability: P(NonNormal) × P(subtype | NonNormal)
        prob_4cls[global_idx, 0] = s1_probs[global_idx, 0]
        prob_4cls[global_idx, 1] = s2_probs[j, 0] * s1_probs[global_idx, 1]
        prob_4cls[global_idx, 2] = s2_probs[j, 1] * s1_probs[global_idx, 1]
        prob_4cls[global_idx, 3] = s2_probs[j, 2] * s1_probs[global_idx, 1]
    else:
        # Stage 1 missed this non-Normal window (predicted Normal)
        # Prediction stays 0 (Normal) from Pass 1 above
        # Log it for analysis — no override here; overriding would require
        # running Stage 2 on ALL windows, not just the pre-selected non-Normals
        pass

# ── Pass 3: handle Stage 1 false alarms (predicted NonNormal, truly Normal)
# For these windows there is no Stage 2 entry. Assign to most probable subtype
# based on Stage 2 prior class frequencies.
s1_false_alarm_idx = np.where((s1_preds == 1) & (y_test_4cls == 0))[0]
if len(s1_false_alarm_idx) > 0:
    # Use Stage 2 class priors (from training distribution) as a soft fallback
    s2_prior = np.array([s2_info.get("val_f1_AF", 0.33),
                          s2_info.get("val_f1_OT", 0.33),
                          s2_info.get("val_f1_NO", 0.33)], dtype=np.float32)
    s2_prior /= s2_prior.sum()
    for idx in s1_false_alarm_idx:
        cls4 = S2_TO_4CLASS[int(s2_prior.argmax())]
        final_preds[idx]  = cls4
        prob_4cls[idx, 0] = s1_probs[idx, 0]
        prob_4cls[idx, 1] = s2_prior[0] * s1_probs[idx, 1]
        prob_4cls[idx, 2] = s2_prior[1] * s1_probs[idx, 1]
        prob_4cls[idx, 3] = s2_prior[2] * s1_probs[idx, 1]

# Normalise so rows sum to 1 (needed for AUC)
row_sums = prob_4cls.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
prob_4cls /= row_sums

print("Final prediction distribution:")
for i, name in enumerate(CLASS_NAMES_4):
    true_count = (y_test_4cls == i).sum()
    pred_count = (final_preds == i).sum()
    print(f"  {name}: pred={pred_count}  true={true_count}")
print(f"\nStage 1 misses (NonNormal predicted as Normal) : {fn}")
print(f"Stage 1 false alarms (Normal predicted as NonNormal): {fp}")


## 11 · Full 4-class evaluation

In [ ]:
print("=" * 65)
print("HIERARCHICAL CLASSIFIER — FULL TEST SET EVALUATION")
print("=" * 65)
print()
print(classification_report(y_test_4cls, final_preds,
                             target_names=CLASS_NAMES_4, digits=4))

macro_f1   = f1_score(y_test_4cls,   final_preds, average="macro")
macro_prec = precision_score(y_test_4cls, final_preds, average="macro", zero_division=0)
macro_rec  = recall_score(y_test_4cls,   final_preds, average="macro", zero_division=0)
acc        = float(np.mean(final_preds == y_test_4cls))
mcc        = matthews_corrcoef(y_test_4cls, final_preds)

f1_n  = f1_score(y_test_4cls, final_preds, labels=[0], average="macro")
f1_af = f1_score(y_test_4cls, final_preds, labels=[1], average="macro")
f1_ot = f1_score(y_test_4cls, final_preds, labels=[2], average="macro")
f1_no = f1_score(y_test_4cls, final_preds, labels=[3], average="macro")

print(f"Overall accuracy  : {acc:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")
print(f"Macro Precision   : {macro_prec:.4f}")
print(f"Macro Recall      : {macro_rec:.4f}")
print(f"MCC               : {mcc:.4f}")
print()
print("Per-class F1 vs targets:")
print(f"  Normal  : {f1_n:.4f}   {'✓ PASS' if f1_n  >= 0.88 else '✗ FAIL'} (target ≥ 0.88)")
print(f"  AF      : {f1_af:.4f}   {'✓ PASS' if f1_af >= 0.80 else '✗ FAIL'} (target ≥ 0.80)")
print(f"  Other   : {f1_ot:.4f}   {'✓ PASS' if f1_ot >= 0.70 else '✗ FAIL'} (target ≥ 0.70)")
print(f"  Noisy   : {f1_no:.4f}   {'✓ PASS' if f1_no >= 0.80 else '✗ FAIL'} (target ≥ 0.80)")
print()
print(f"Overall target (macro F1 ≥ 0.88): {'✓ PASS' if macro_f1 >= 0.88 else '✗ FAIL'} ({macro_f1:.4f})")

try:
    auc = roc_auc_score(y_test_4cls, prob_4cls, multi_class="ovr", average="macro")
    print(f"Macro AUC (OvR)   : {auc:.4f}")
except Exception as e:
    print(f"AUC: {e}")


## 12 · Confusion matrices

In [ ]:
fig = plt.figure(figsize=(15, 6))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# ── 4-class confusion matrix ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
cm4 = confusion_matrix(y_test_4cls, final_preds)
im1 = ax1.imshow(cm4, cmap="Blues")
ax1.set_xticks(range(4)); ax1.set_yticks(range(4))
ax1.set_xticklabels(CLASS_NAMES_4, rotation=20, ha="right")
ax1.set_yticklabels(CLASS_NAMES_4)
ax1.set_xlabel("Predicted"); ax1.set_ylabel("True")
ax1.set_title("4-class Confusion Matrix (Test Set)")
for i in range(4):
    for j in range(4):
        pct = cm4[i, j] / max(cm4[i].sum(), 1) * 100
        ax1.text(j, i, f"{cm4[i,j]}\n({pct:.1f}%)", ha="center", va="center",
                 color="white" if cm4[i, j] > cm4.max() / 2 else "black", fontsize=9)
plt.colorbar(im1, ax=ax1)

# ── Stage 1 binary confusion matrix ──────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
cm2 = confusion_matrix(s1_true, s1_preds)
im2 = ax2.imshow(cm2, cmap="Oranges")
ax2.set_xticks([0, 1]); ax2.set_yticks([0, 1])
ax2.set_xticklabels(["Normal", "NonNormal"])
ax2.set_yticklabels(["Normal", "NonNormal"])
ax2.set_xlabel("Predicted"); ax2.set_ylabel("True")
ax2.set_title("Stage 1 Binary Confusion Matrix")
for i in range(2):
    for j in range(2):
        pct = cm2[i, j] / max(cm2[i].sum(), 1) * 100
        ax2.text(j, i, f"{cm2[i,j]}\n({pct:.1f}%)", ha="center", va="center",
                 color="white" if cm2[i, j] > cm2.max() / 2 else "black", fontsize=11)
plt.colorbar(im2, ax=ax2)

plt.savefig(os.path.join(OUT_DIR, "confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices.png")


## 13 · Per-class F1 bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 4-class F1 bars ───────────────────────────────────────────────────────
f1s     = [f1_n, f1_af, f1_ot, f1_no]
targets = [0.88, 0.80,  0.70,  0.80]
colors  = ["steelblue", "coral", "seagreen", "orchid"]

bars = axes[0].bar(CLASS_NAMES_4, f1s, color=colors, edgecolor="white", linewidth=1.2)
for bar, f, t in zip(bars, f1s, targets):
    axes[0].axhline(t, color=bar.get_facecolor(), linestyle="--", alpha=0.5, linewidth=1)
    axes[0].text(bar.get_x() + bar.get_width()/2, f + 0.01, f"{f:.4f}",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("F1 Score"); axes[0].set_title("Per-class F1 (dashed = target)")
axes[0].grid(axis="y", alpha=0.3)

# ── Stage F1 comparison ───────────────────────────────────────────────────
stage_labels = ["Stage 1\n(binary)", "Stage 2\n(3-class)", "Final\n(4-class)"]
stage_f1s    = [s1_f1, s2_f1, macro_f1]
scolors      = ["#4472C4", "#ED7D31", "#70AD47"]
bars2 = axes[1].bar(stage_labels, stage_f1s, color=scolors, edgecolor="white", linewidth=1.2)
for bar, f in zip(bars2, stage_f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, f + 0.01, f"{f:.4f}",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")
axes[1].axhline(0.88, color="red", linestyle="--", linewidth=1.2, label="Target 0.88")
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("Macro F1")
axes[1].set_title("Stage-by-stage macro F1"); axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "f1_breakdown.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved: f1_breakdown.png")


## 14 · Error flow analysis

In [ ]:
total = n_total

# Stage 1 errors
s1_miss_count  = int(fn)  # NonNormal predicted Normal
s1_alarm_count = int(fp)  # Normal predicted NonNormal

# Stage 2 errors (on true non-Normal, Stage-1-correct windows)
s1_correct_nn    = np.where((s1_preds == 1) & (y_test_4cls != 0))[0]
s2_window_idx    = []
true_nn_list     = list(true_nonnormal_idx)
for g in s1_correct_nn:
    if g in true_nn_list:
        s2_window_idx.append(true_nn_list.index(g))

if s2_window_idx:
    s2_subset_preds = s2_preds[s2_window_idx]
    s2_subset_true  = y_test_s2[s2_window_idx]
    s2_misclassified = int((s2_subset_preds != s2_subset_true).sum())
else:
    s2_misclassified = int((s2_preds != y_test_s2).sum())

total_errors  = int((final_preds != y_test_4cls).sum())
correct       = total - total_errors

print("Error flow analysis:")
print(f"  Total test windows         : {total}")
print(f"  Correctly classified       : {correct}  ({correct/total*100:.1f}%)")
print(f"  Total errors               : {total_errors}  ({total_errors/total*100:.1f}%)")
print()
print(f"  ├── Stage 1 misses         : {s1_miss_count}  (true NonNormal → predicted Normal)")
print(f"  │       These windows never reach Stage 2")
print(f"  ├── Stage 1 false alarms   : {s1_alarm_count}  (true Normal → predicted NonNormal)")
print(f"  │       Sent to Stage 2 but no spectrogram available (edge case)")
print(f"  └── Stage 2 misclassifications : {s2_misclassified}  (wrong AF/Other/Noisy label)")
print()
print(f"  Stage 1 sensitivity : {tp/(fn+tp):.4f}  (how many NonNormals are caught)")
print(f"  Stage 1 specificity : {tn/(tn+fp):.4f}  (how many Normals are spared)")


## 15 · Save summary JSON

In [ ]:
summary = {
    "macro_f1":              float(macro_f1),
    "macro_precision":       float(macro_prec),
    "macro_recall":          float(macro_rec),
    "mcc":                   float(mcc),
    "accuracy":              float(acc),
    "f1_Normal":             float(f1_n),
    "f1_AF":                 float(f1_af),
    "f1_Other":              float(f1_ot),
    "f1_Noisy":              float(f1_no),
    "stage1_binary_macro_f1": float(s1_f1),
    "stage1_f1_Normal":      float(s1_f1_N),
    "stage1_f1_NonNormal":   float(s1_f1_NN),
    "stage1_sensitivity":    float(tp / (fn + tp)),
    "stage1_specificity":    float(tn / (tn + fp)),
    "stage2_macro_f1":       float(s2_f1),
    "stage1_threshold":      float(S1_THRESHOLD),
    "stage1_false_alarms":   int(fp),
    "stage1_misses":         int(fn),
    "total_errors":          int(total_errors),
    "total_windows":         int(total),
    "pass_Normal":           bool(f1_n  >= 0.88),
    "pass_AF":               bool(f1_af >= 0.80),
    "pass_Other":            bool(f1_ot >= 0.70),
    "pass_Noisy":            bool(f1_no >= 0.80),
    "pass_overall":          bool(macro_f1 >= 0.88),
}
try:
    summary["macro_auc_ovr"] = float(auc)
except NameError:
    pass

with open(os.path.join(OUT_DIR, "phase3_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Saved: phase3_summary.json")

print()
print("=" * 65)
print("PHASE 3 COMPLETE")
print("=" * 65)
print(f"  Macro F1   : {macro_f1:.4f}  {'✓ PASS (≥ 0.88)' if macro_f1 >= 0.88 else '✗ FAIL (< 0.88)'}")
print(f"  MCC        : {mcc:.4f}")
print(f"  Accuracy   : {acc:.4f}")
print()
print("Output files saved to:", OUT_DIR)
for fname in sorted(os.listdir(OUT_DIR)):
    print(f"  {fname}")
